In [46]:
import pandas as pd
from rdkit import Chem
from dataclasses import dataclass
from typing import Optional

In [ ]:
AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DNA  = set("ACGT")
RNA  = set("ACGU")
NUC  = set("ACGTU")

NUC_AMBIG = set("NRYSWKMBDHVX")
NUC_LIKE  = NUC | NUC_AMBIG

@dataclass(frozen=True)
class SeqCheck:
    kind: str                 # "DNA"|"RNA"|"NUCLEIC_MIXED"|"NUCLEIC_AMBIG"|"AA"|"SMILES"|"INVALID"
    length: int
    invalid_chars: str
    canonical: Optional[str]

def try_canonical_smiles(s: str) -> Optional[str]:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

def classify_sequence(s: str, allow_aa_uo: bool = False) -> SeqCheck:
    if s is None:
        return SeqCheck("INVALID", 0, "None", None)

    s = s.strip()
    if not s:
        return SeqCheck("INVALID", 0, "EMPTY", None)

    s_smiles = s
    s_seq = s.upper()
    chars = set(s_seq)

    if chars <= NUC_LIKE:
        amb = chars & NUC_AMBIG
        if amb:
            return SeqCheck("INVALID", len(s_seq), "".join(sorted(amb)), None)

        # strictly A/C/G/T/U only
        has_t = "T" in chars
        has_u = "U" in chars
        if has_t and has_u:
            return SeqCheck("NUCLEIC_MIXED", len(s_seq), "", None)
        if has_t:
            return SeqCheck("DNA", len(s_seq), "", None)
        if has_u:
            return SeqCheck("RNA", len(s_seq), "", None)
        return SeqCheck("NUCLEIC_AMBIG", len(s_seq), "", None)  # only A/C/G

    # 2) AA
    AA = AA20 | (set("UO") if allow_aa_uo else set())
    if chars <= AA:
        return SeqCheck("AA", len(s_seq), "", None)

    # 3) SMILES
    can = try_canonical_smiles(s_smiles)
    if can is not None:
        return SeqCheck("SMILES", len(s_smiles), "", can)

    # 4) INVALID
    allowed = AA | NUC  
    invalid = "".join(sorted(chars - allowed))
    return SeqCheck("INVALID", len(s_seq), invalid, None)

In [ ]:
base = "data"
binding_db_interactions_path = f"{base}/processed_binding_db/binding_db_interactions_unique.csv"
binding_db_molecules_path = f"{base}/processed_binding_db/binding_db_molecules.csv"
binding_db_proteins_path = f"{base}/processed_binding_db/binding_db_proteins.csv"

biogrid_ppi_unique_path = f"{base}/processed_biogrid/biogrid_ppi_unique.csv"
biogrid_name_to_sequence_path = f"{base}/processed_biogrid/name_to_sequences.csv"
biogrid_sequence_to_name_path = f"{base}/processed_biogrid/sequence_to_names.csv"

In [48]:
binding_db_interactions = pd.read_csv(binding_db_interactions_path)
binding_db_molecules = pd.read_csv(binding_db_molecules_path)
binding_db_proteins = pd.read_csv(binding_db_proteins_path)

In [82]:
biogrid_ppi_unique = pd.read_csv(biogrid_ppi_unique_path)
biogrid_name_to_sequence = pd.read_csv(biogrid_name_to_sequence_path)
biogrid_sequence_to_name = pd.read_csv(biogrid_sequence_to_name_path)

/var/folders/_3/p73rghg16cg5d9z62wt7_w1w0000gn/T/ipykernel_7520/3314343483.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid_ppi_unique = pd.read_csv(biogrid_ppi_unique_path)


In [6]:
binding_db_interactions

,protein_name,protein_sequence,molecule_name,molecule_smiles,Target Source Organism According to Curator or DataSource,Ki (nM),IC50 (nM),Kd (nM),EC50 (nM),pH,Temp C,PubChem CID of Ligand,UniProt (SwissProt) Primary ID of Target Chain
0,Gag-Pol polyprotein,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,Human immunodeficiency virus,0.016,NaN,0.016,NaN,4.7,25.0,53308627.0,Q9QBY3
1,Cytochrome P450 3A4,MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPF...,"US9447092, 3",Cc1nc(CN2CCN(c3c(Cl)cnc4[nH]c(-c5cn(C)nc5C)nc3...,Homo sapiens,NaN,>50000,NaN,NaN,NaN,NaN,71463198.0,P08684
2,Galactokinase,MAALRQPQVAELLAEARRAFREEFGAEPELAVSAPGRVNLIGEHTD...,"US9447087, 24::2-(benzo[d]oxazol-2-ylamino)-4'...",O=C1CCCC2=C1C1(CCS(=O)(=O)C1)N=C(Nc1nc3ccccc3o...,Homo sapiens,NaN,6676.9,NaN,NaN,NaN,NaN,44640149.0,P51570
3,Gag-Pol polyprotein,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,Human immunodeficiency virus,0.005,NaN,0.005,NaN,4.7,25.0,53308628.0,Q9QBY3
4,Caspase-3,MENTENSVDSKSIKNLEPKIIHGSESMDSGISLDNSYKMDYPEMGL...,5-({[(5-{[(2S)-1-carboxy-4-{[(2-chlorophenyl)m...,CN(Cc1ccc(O)c(C(=O)O)c1)Cc1ccc(C(=O)N[C@@H](CC...,Homo sapiens,90,NaN,90.000,NaN,7.4,25.0,5327301.0,P42574
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2253655,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5397509,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.600000,NaN,NaN,NaN,NaN,170227690.0,Q9NWZ3
2253656,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5410702,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.300000,NaN,NaN,NaN,NaN,170227701.0,Q9NWZ3
2253657,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5421046,COc1cc2nn([C@H]3CC[C@]4(CC3)COC(=O)N4C)cc2cc1C...,Homo sapiens,NaN,0.400000,NaN,NaN,NaN,NaN,169132490.0,Q9NWZ3
2253658,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5408807,COc1cc2nn([C@H]3CC[C@H](n4ccccc4=O)CC3)cc2cc1C...,Homo sapiens,NaN,0.500000,NaN,NaN,NaN,NaN,170227781.0,Q9NWZ3


In [28]:
a = (
    binding_db_interactions
        [["protein_name", "protein_sequence"]]
        .rename(columns={
            "protein_name": "name",
            "protein_sequence": "content"
        })
)

b = (
    binding_db_interactions
        [["molecule_name", "molecule_smiles"]]
        .rename(columns={
            "molecule_name": "name",
            "molecule_smiles": "content"
        })
)

binding_db = pd.concat([a, b])

In [ ]:
res = a["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
AA    2253660
Name: count, dtype: int64

In [30]:
mask_AA = res.apply(lambda x: x.kind) == "AA"
AA_rows = a.loc[mask_AA, ["content", "name"]].copy()
AA_rows["invalid_chars"] = res[mask_AA].apply(lambda x: x.invalid_chars)
AA_rows["len"] = res[mask_AA].apply(lambda x: x.length)
AA_rows = AA_rows.reset_index()
AA_rows = AA_rows[["index", "content", "name", "invalid_chars", "len"]]
AA_rows.to_csv("binding_db_a_AA.csv")
AA_rows

,index,content,name,invalid_chars,len
0,0,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,Gag-Pol polyprotein,,99
1,1,MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPF...,Cytochrome P450 3A4,,503
2,2,MAALRQPQVAELLAEARRAFREEFGAEPELAVSAPGRVNLIGEHTD...,Galactokinase,,392
3,3,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,Gag-Pol polyprotein,,99
4,4,MENTENSVDSKSIKNLEPKIIHGSESMDSGISLDNSYKMDYPEMGL...,Caspase-3,,277
...,...,...,...,...,...
2253655,2253655,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Interleukin-1 receptor-associated kinase 4,,460
2253656,2253656,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Interleukin-1 receptor-associated kinase 4,,460
2253657,2253657,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Interleukin-1 receptor-associated kinase 4,,460
2253658,2253658,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Interleukin-1 receptor-associated kinase 4,,460


In [ ]:
res = b["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
SMILES     2253497
INVALID        156
AA               7
Name: count, dtype: int64

In [ ]:
mask_AA = res.apply(lambda x: x.kind) == "AA"
AA_rows = b.loc[mask_AA, ["content", "name"]].copy()
AA_rows["invalid_chars"] = res[mask_AA].apply(lambda x: x.invalid_chars)
AA_rows["len"] = res[mask_AA].apply(lambda x: x.length)
AA_rows = AA_rows.reset_index()
AA_rows = AA_rows[["index", "content", "name", "invalid_chars", "len"]]
AA_rows

,index,content,name,invalid_chars,len
0,1105135,CI,CHEMBL3228330,,2
1,2003634,F,CHEBI:29228::Fluoride,,1
2,2003635,Cl,CHEBI:17883::E507::Hydrochloric Acid,,2
3,2003638,F,CHEBI:29228::Fluoride,,1
4,2003639,Cl,CHEBI:17883::E507::Hydrochloric Acid,,2
5,2003640,I,CHEBI:43451::Iodide,,1
6,2003643,I,CHEBI:43451::Iodide,,1


In [12]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = b.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
INVALID_rows

,index,content,name,invalid_chars,len
0,5699,NCCS,2-aminoethane-1-thiol::Cysteamine,NS,4
1,5703,NCCN,"ethane-1,2-diamine::CHEMBL816::ethylenediamine",N,4
2,114196,NCCCNCCCCNCCCN,"N,N''-bis(3-azanylpropyl)butane-1,4-diamine;hy...",N,14
3,116208,NCCCNCCCCNCCCN,"N,N''-bis(3-azanylpropyl)butane-1,4-diamine;hy...",N,14
4,133669,NCCCCNCCCN,"spermidine::4-azaoctane-1,8-diamine::4-azaocta...",N,10
...,...,...,...,...,...
151,2101833,CCCCCCCCCCCCN,CHEMBL109904::Dodecylamine,N,13
152,2136349,NCCSSCCN,CYSTAMINE DIHYDROCHLORIDE,NS,8
153,2186603,NCCCCNCCCN,"1,5,10-triazadecane::4-azaoctamethylenediamine...",N,10
154,2206526,NCCCCNCCCN,"1,5,10-triazadecane::4-azaoctamethylenediamine...",N,10


In [14]:
INVALID_rows["invalid_chars"].value_counts()

invalid_chars
N     134
NS     13
S       7
BR      2
Name: count, dtype: int64

In [13]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = b.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows

,index,content,name,invalid_chars,len
0,0,CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",,91
1,1,Cc1nc(CN2CCN(c3c(Cl)cnc4[nH]c(-c5cn(C)nc5C)nc3...,"US9447092, 3",,55
2,2,O=C1CCCC2=C1C1(CCS(=O)(=O)C1)N=C(Nc1nc3ccccc3o...,"US9447087, 24::2-(benzo[d]oxazol-2-ylamino)-4'...",,50
3,3,CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",,92
4,4,CN(Cc1ccc(O)c(C(=O)O)c1)Cc1ccc(C(=O)N[C@@H](CC...,5-({[(5-{[(2S)-1-carboxy-4-{[(2-chlorophenyl)m...,,73
...,...,...,...,...,...
2253492,2253655,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,CHEMBL5397509,,77
2253493,2253656,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,CHEMBL5410702,,83
2253494,2253657,COc1cc2nn([C@H]3CC[C@]4(CC3)COC(=O)N4C)cc2cc1C...,CHEMBL5421046,,64
2253495,2253658,COc1cc2nn([C@H]3CC[C@H](n4ccccc4=O)CC3)cc2cc1C...,CHEMBL5408807,,64


In [21]:
mask = (INVALID_rows["content"] == "S") | (INVALID_rows["invalid_chars"] == "BR")

SMILES_rows = pd.concat(
    [SMILES_rows, INVALID_rows.loc[~mask]]
)

INVALID_rows = INVALID_rows.loc[mask]

In [26]:
SMILES_rows
SMILES_rows.to_csv("binding_db_b_SMILES.csv")

In [27]:
INVALID_rows = pd.concat(
    [AA_rows, INVALID_rows]
)
INVALID_rows.to_csv("binding_db_b_INVALID.csv")

In [31]:
biogrid_ppi_unique

,SWISS-PROT Accessions Interactor A,SWISS-PROT Accessions Interactor B,Score,Qualifications,protein_name_a,protein_sequence_a,protein_name_b,protein_sequence_b
0,P45985,Q14315,-,-,Dual specificity mitogen-activated protein kin...,MAAPSPSGGGGSGGGSGSGTPGPVGSPAPGHPAVSSMQGKRKALKL...,Filamin-C,MMNNSGYSDAGLGLGDETDEMPSTEKDLAEDAPWKKIQQNTFTRWC...
1,Q86TC9,P35609,-,-,Myopalladin,MQDDSIEASTSISQLLRESYLAETRHRGNNERSRAEPSSNPCHFGS...,Alpha-actinin-2,MNQIEPGVQYNYVYDEDEYMIQEEEWDRDLLLDPAWEKQQRKTFTA...
2,Q04771,P49354,-,-,Activin receptor type-1,MVDGVMILPVLIMIALPSPSMEDEKPKVNPKLYMCVCEGLSCGNED...,Protein farnesyltransferase/geranylgeranyltran...,MAATEGVGEAAQGGEPGQPAQPPPQPHPPPPQQQHKEEMAAEAGEA...
3,P23769,P29590,-,-,Endothelial transcription factor GATA-2,MEVAPEQPRWMAHPAVLNAQHPDSHHPGLAHNYMEPAQLLPPDEVD...,Protein PML,MEPAPARSPRPQQDPARPQEPTMPPPETPSEGRQPSPSPSPTERAP...
4,P15927,P40763,-,-,Replication protein A 32 kDa subunit,MWNSGFESYGSSSYGGAGGYTQSPGGFGSPAPSQAEKKSRARAQHI...,Signal transducer and activator of transcripti...,MAQWNQLQQLDTRYLEQLHQLYSDSFPMELRQFLAPWIESQDWAYA...
...,...,...,...,...,...,...,...,...
2095385,Q13085,Q8NHY2,-,-,Acetyl-CoA carboxylase 1,MDEPSPLAQPLELNQHSRFIIGSVSEDNSEDEISNLVKLDLLEEKE...,E3 ubiquitin-protein ligase COP1,MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSVAVSAAA...
2095386,P53396,Q86T96,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase RNF180,MKRSKELITKNHSQEETSILRCWKCRKCIASSGCFMEYLENQVIKD...
2095387,P53396,Q86TM6,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase synoviolin,MFRTAVMMAASLALTGAVVAHAYYLKHQFYPTVVYLTKSSPSMAVL...
2095388,P53396,P46934,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase NEDD4,MAQSLRLHFAARRSNTYPLSETSGDDLDSHVHMCFKRPTRISTSNV...


In [32]:
a = (
    biogrid_ppi_unique
        [["protein_name_a", "protein_sequence_a"]]
        .rename(columns={
            "protein_name_a": "name",
            "protein_sequence_a": "content"
        })
)

b = (
    biogrid_ppi_unique
        [["protein_name_b", "protein_sequence_b"]]
        .rename(columns={
            "protein_name_b": "name",
            "protein_sequence_b": "content"
        })
)

biogrid_ppi = pd.concat([a, b])

In [ ]:
res = biogrid_ppi["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

[18:31:02] SMILES Parse Error: syntax error while parsing: MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLLSRSKSTRGEWRRMLTSEGLRCVWKSFLLDAYKQVKLGEDAPNSSVVHVSSTEGGDNSGNGTQEKIAEGATCHLLDFASPERPLVVNFGSATUPPFTSQLPAFRKLVEEFSSVADFLLVYIDEAHPSDGWAIPGDSSLSFEVKKHQNQEDRCAAAQQLLERFSLPPQCRVVADRMDNNANIAYGVAFERVCIVQRQKIAYLGGKGPFSYNLQEVRHWLEKNFSKRUKKTRLAG
[18:31:02] SMILES Parse Error: check for mistakes around position 1:
[18:31:02] MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLL
[18:31:02] ^
[18:31:02] SMILES Parse Error: Failed parsing SMILES 'MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLLSRSKSTRGEWRRMLTSEGLRCVWKSFLLDAYKQVKLGEDAPNSSVVHVSSTEGGDNSGNGTQEKIAEGATCHLLDFASPERPLVVNFGSATUPPFTSQLPAFRKLVEEFSSVADFLLVYIDEAHPSDGWAIPGDSSLSFEVKKHQNQEDRCAAAQQLLERFSLPPQCRVVADRMDNNANIAYGVAFERVCIVQRQKIAYLGGKGPFSYNLQEVRHWLEKNFSKRUKKTRLAG' for input: 'MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLLSRSKSTRGEWRRMLTSEGLRCVWKSFLLDAYKQVKLGEDAPNSSVVHVSSTEGGDNSGNGTQEKIAEGATCHLLDFASPERPLVVNFGSATUPPFTSQLPAFRKLVEEFSSVADFLLVYIDEAHPSDGWAIPGDSSLSFEVKKHQNQEDR

content
AA         4189137
INVALID       1643
Name: count, dtype: int64

In [ ]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = biogrid_ppi.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
INVALID_rows

,index,content,name,invalid_chars,len
0,821,MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLLSRSKS...,Type II iodothyronine deiodinase,,273
1,1081,MAPRRNSNISSSGNSTQQQHQQQLQQQHQTQQHQLLQFYAENVNSS...,Headcase protein,X,1080
2,1449,MAFLCAPGLTFFLTYSIFSAVLGMLQFGYNTGVINAPEKNIENFMK...,Glucose transporter type 1,X,1440
3,1450,MAFLCAPGLTFFLTYSIFSAVLGMLQFGYNTGVINAPEKNIENFMK...,Glucose transporter type 1,X,1440
4,1601,MSASASACDCLVGVPTGPTLASTCGGSAFMLFMGLLEVFIRSQCDL...,"Glucose dehydrogenase [FAD, quinone]",,625
...,...,...,...,...,...
1638,2091497,MVYISNGQVLDSRSQSPWRLSLITDFFWGIAEFVVLFFKTLLQQDV...,Selenoprotein K,,94
1639,2091576,MAVYRAALGASLAAARLLPLGRCSPSPAPRSTLSGAAMEPAPRWLA...,"Protein adenylyltransferase SelO, mitochondrial",,669
1640,2091672,MSLGRLCRLLKPALLCGALAAPGLAGTMCASRDDWRCARSMHEFSA...,Phospholipid hydroperoxide glutathione peroxid...,,197
1641,2093696,MVAMAAGPSGCLVPAFGLRLLLATVLQAVSAFGAEFSSEACRELGF...,Selenoprotein F,,165


In [35]:
INVALID_rows['invalid_chars'].value_counts()

invalid_chars
      1209
X      413
BX      21
Name: count, dtype: int64

In [36]:
mask = ~INVALID_rows["invalid_chars"].isin(["X", "BX"])

subset = INVALID_rows[mask]
rest   = INVALID_rows[~mask]

In [ ]:
mask_AA = res.apply(lambda x: x.kind) == "AA"
AA_rows = biogrid_ppi.loc[mask_AA, ["content", "name"]].copy()
AA_rows["invalid_chars"] = res[mask_AA].apply(lambda x: x.invalid_chars)
AA_rows["len"] = res[mask_AA].apply(lambda x: x.length)
AA_rows = AA_rows.reset_index()
AA_rows = AA_rows[["index", "content", "name", "invalid_chars", "len"]]
AA_rows

,index,content,name,invalid_chars,len
0,0,MAAPSPSGGGGSGGGSGSGTPGPVGSPAPGHPAVSSMQGKRKALKL...,Dual specificity mitogen-activated protein kin...,,399
1,1,MQDDSIEASTSISQLLRESYLAETRHRGNNERSRAEPSSNPCHFGS...,Myopalladin,,1320
2,2,MVDGVMILPVLIMIALPSPSMEDEKPKVNPKLYMCVCEGLSCGNED...,Activin receptor type-1,,509
3,3,MEVAPEQPRWMAHPAVLNAQHPDSHHPGLAHNYMEPAQLLPPDEVD...,Endothelial transcription factor GATA-2,,480
4,4,MWNSGFESYGSSSYGGAGGYTQSPGGFGSPAPSQAEKKSRARAQHI...,Replication protein A 32 kDa subunit,,270
...,...,...,...,...,...
4189132,2095385,MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSVAVSAAA...,E3 ubiquitin-protein ligase COP1,,731
4189133,2095386,MKRSKELITKNHSQEETSILRCWKCRKCIASSGCFMEYLENQVIKD...,E3 ubiquitin-protein ligase RNF180,,592
4189134,2095387,MFRTAVMMAASLALTGAVVAHAYYLKHQFYPTVVYLTKSSPSMAVL...,E3 ubiquitin-protein ligase synoviolin,,617
4189135,2095388,MAQSLRLHFAARRSNTYPLSETSGDDLDSHVHMCFKRPTRISTSNV...,E3 ubiquitin-protein ligase NEDD4,,1319


In [38]:
AA_rows = pd.concat(
    [AA_rows, subset]
)
AA_rows.to_csv("biogrid_ppi_AA.csv")

In [45]:
rest.to_csv("biogrid_ppi_INVALID.csv")

In [59]:
def try_canonical_smiles(s: str):
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

In [60]:
binding_db_interactions['molecule_smiles'].apply(try_canonical_smiles)

0          CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...
1          Cc1nc(CN2CCN(c3c(Cl)cnc4[nH]c(-c5cn(C)nc5C)nc3...
2          O=C1CCCC2=C1C1(CCS(=O)(=O)C1)N=C(Nc1nc3ccccc3o...
3          CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...
4          CN(Cc1ccc(O)c(C(=O)O)c1)Cc1ccc(C(=O)N[C@@H](CC...
                                 ...                        
2253655    COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...
2253656    COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...
2253657    COc1cc2nn([C@H]3CC[C@]4(CC3)COC(=O)N4C)cc2cc1C...
2253658    COc1cc2nn([C@H]3CC[C@H](n4ccccc4=O)CC3)cc2cc1C...
2253659    COc1cc2nn([C@H]3CC[C@H](n4ccnc4C(C)O)CC3)cc2cc...
Name: molecule_smiles, Length: 2253660, dtype: object

In [ ]:
FORBIDDEN = {9, 17, 35, 53, 16}  # F, Cl, Br, I, S

def has_forbidden_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return True  
    return any(a.GetAtomicNum() in FORBIDDEN for a in mol.GetAtoms())

mask = binding_db_interactions['molecule_smiles'].apply(has_forbidden_atoms)

removed = binding_db_interactions[mask]
binding_db_interactions_2 = binding_db_interactions[~mask]

In [65]:
pattern = r'Cl|Br|F|I|S'

binding_db_interactions_3 = binding_db_interactions[
    ~binding_db_interactions['molecule_smiles'].str.contains(pattern, na=False)
]


In [66]:
binding_db_interactions_3

,protein_name,protein_sequence,molecule_name,molecule_smiles,Target Source Organism According to Curator or DataSource,Ki (nM),IC50 (nM),Kd (nM),EC50 (nM),pH,Temp C,PubChem CID of Ligand,UniProt (SwissProt) Primary ID of Target Chain,check_prot_seq
109,Caspase-3,MENTENSVDSKSIKNLEPKIIHGSESMDSGISLDNSYKMDYPEMGL...,(4S)-4-{[(1S)-1-{[(2S)-1-carboxy-3-oxopropan-2...,CC(=O)N[C@@H](CC(=O)O)C(=O)N[C@@H](CCC(=O)O)C(...,Homo sapiens,0.23,NaN,0.23,NaN,7.5,25.0,644345.0,P42574,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
110,Caspase-1,MADKVLKEKRKLFIRSMGEGTINGLLDELLQTRVLNKEEMEKVKRE...,(4S)-4-{[(1S)-1-{[(2S)-1-carboxy-3-oxopropan-2...,CC(=O)N[C@@H](CC(=O)O)C(=O)N[C@@H](CCC(=O)O)C(...,Homo sapiens,18,NaN,18.00,NaN,7.5,25.0,644345.0,P29466,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
163,11-beta-hydroxysteroid dehydrogenase 1,MAFMKKYLLPILGLFMAYYYYSANEEFRPEMLQGKKVIVTGASKGI...,"US8497281, 101",CNC(=O)c1ccc2c(c1)C[C@H]1[C@@H]2CCCN1C(=O)c1cc...,Homo sapiens,NaN,712,NaN,NaN,NaN,NaN,67223164.0,P28845,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
164,Focal adhesion kinase 1,MAAAYLDPNLNHTPNSSTKTHLGTGMERSPGAMERVLKVFHYFESN...,"US8501936, 298",Cn1ncc2c(-c3cccn4nc(Nc5ccc6c(c5)CCNCC6)nc34)cc...,Homo sapiens,NaN,1033.14,NaN,NaN,NaN,NaN,49838143.0,Q05397,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
166,Tyrosine-protein kinase JAK2,MGMACLTMTEMEGTSTSSIYQNGDISGNANSMKQIDPVLQVYLYHS...,"US8501936, 298::US8501936, 300",Cn1ncc2c(-c3cccn4nc(Nc5ccc6c(c5)CCN(C(=O)OC(C)...,Homo sapiens,NaN,16.54,NaN,NaN,NaN,NaN,59612433.0,O60674,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2253655,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5397509,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.600000,NaN,NaN,NaN,NaN,170227690.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253656,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5410702,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.300000,NaN,NaN,NaN,NaN,170227701.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253657,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5421046,COc1cc2nn([C@H]3CC[C@]4(CC3)COC(=O)N4C)cc2cc1C...,Homo sapiens,NaN,0.400000,NaN,NaN,NaN,NaN,169132490.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253658,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5408807,COc1cc2nn([C@H]3CC[C@H](n4ccccc4=O)CC3)cc2cc1C...,Homo sapiens,NaN,0.500000,NaN,NaN,NaN,NaN,170227781.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."


In [50]:
binding_db_interactions['check_prot_seq'] = binding_db_interactions['protein_sequence'].map(set)

In [51]:
s = "ACDEFGHIKLMNPQRSTVWY"
chars = set(s)
chars.add("U")
chars.add("O")

ALLOWED = chars

mask = binding_db_interactions['check_prot_seq'].apply(lambda s: s.issubset(ALLOWED))

AA20X = set("ACDEFGHIKLMNPQRSTVWYX")

mask_protein = binding_db_interactions['check_prot_seq'].apply(
    lambda s: s.issubset(AA20X)
)

invalid = binding_db_interactions[~mask]
binding_db_interactions_1 = binding_db_interactions[mask]

In [63]:
binding_db_interactions_1

,protein_name,protein_sequence,molecule_name,molecule_smiles,Target Source Organism According to Curator or DataSource,Ki (nM),IC50 (nM),Kd (nM),EC50 (nM),pH,Temp C,PubChem CID of Ligand,UniProt (SwissProt) Primary ID of Target Chain,check_prot_seq
0,Gag-Pol polyprotein,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,Human immunodeficiency virus,0.016,NaN,0.016,NaN,4.7,25.0,53308627.0,Q9QBY3,"{F, W, A, D, I, N, G, E, M, K, P, L, Q, C, R, ..."
1,Cytochrome P450 3A4,MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPF...,"US9447092, 3",Cc1nc(CN2CCN(c3c(Cl)cnc4[nH]c(-c5cn(C)nc5C)nc3...,Homo sapiens,NaN,>50000,NaN,NaN,NaN,NaN,71463198.0,P08684,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2,Galactokinase,MAALRQPQVAELLAEARRAFREEFGAEPELAVSAPGRVNLIGEHTD...,"US9447087, 24::2-(benzo[d]oxazol-2-ylamino)-4'...",O=C1CCCC2=C1C1(CCS(=O)(=O)C1)N=C(Nc1nc3ccccc3o...,Homo sapiens,NaN,6676.9,NaN,NaN,NaN,NaN,44640149.0,P51570,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
3,Gag-Pol polyprotein,PQITLWKRPIVTVKIGGQLREALLDTGADDTVLEDINLPGKWKPKM...,"(3R,3aS,6aR)-Hexahydrofuro[2,3-b]furan-3-yl ((...",CC[C@H](C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O...,Human immunodeficiency virus,0.005,NaN,0.005,NaN,4.7,25.0,53308628.0,Q9QBY3,"{F, W, A, D, I, N, G, E, M, K, P, L, Q, C, R, ..."
4,Caspase-3,MENTENSVDSKSIKNLEPKIIHGSESMDSGISLDNSYKMDYPEMGL...,5-({[(5-{[(2S)-1-carboxy-4-{[(2-chlorophenyl)m...,CN(Cc1ccc(O)c(C(=O)O)c1)Cc1ccc(C(=O)N[C@@H](CC...,Homo sapiens,90,NaN,90.000,NaN,7.4,25.0,5327301.0,P42574,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2253655,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5397509,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.600000,NaN,NaN,NaN,NaN,170227690.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253656,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5410702,COc1cc2nn([C@H]3CC[C@H](N(C)C(=O)[C@@H](C)O)CC...,Homo sapiens,NaN,0.300000,NaN,NaN,NaN,NaN,170227701.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253657,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5421046,COc1cc2nn([C@H]3CC[C@]4(CC3)COC(=O)N4C)cc2cc1C...,Homo sapiens,NaN,0.400000,NaN,NaN,NaN,NaN,169132490.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2253658,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,CHEMBL5408807,COc1cc2nn([C@H]3CC[C@H](n4ccccc4=O)CC3)cc2cc1C...,Homo sapiens,NaN,0.500000,NaN,NaN,NaN,NaN,170227781.0,Q9NWZ3,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."


In [78]:
binding_db_interactions_2.drop(columns=['check_prot_seq'], inplace=True)
binding_db_interactions_1.drop(columns=['check_prot_seq'], inplace=True)

In [79]:
df_intersection = binding_db_interactions_1.merge(binding_db_interactions_2, how="inner")

In [81]:
df_intersection.to_csv("final_binding_db_interactions.csv", index=False)

In [87]:
biogrid_ppi_unique

,SWISS-PROT Accessions Interactor A,SWISS-PROT Accessions Interactor B,Score,Qualifications,protein_name_a,protein_sequence_a,protein_name_b,protein_sequence_b,check_prot_seq_a,check_prot_seq_b
0,P45985,Q14315,-,-,Dual specificity mitogen-activated protein kin...,MAAPSPSGGGGSGGGSGSGTPGPVGSPAPGHPAVSSMQGKRKALKL...,Filamin-C,MMNNSGYSDAGLGLGDETDEMPSTEKDLAEDAPWKKIQQNTFTRWC...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
1,Q86TC9,P35609,-,-,Myopalladin,MQDDSIEASTSISQLLRESYLAETRHRGNNERSRAEPSSNPCHFGS...,Alpha-actinin-2,MNQIEPGVQYNYVYDEDEYMIQEEEWDRDLLLDPAWEKQQRKTFTA...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2,Q04771,P49354,-,-,Activin receptor type-1,MVDGVMILPVLIMIALPSPSMEDEKPKVNPKLYMCVCEGLSCGNED...,Protein farnesyltransferase/geranylgeranyltran...,MAATEGVGEAAQGGEPGQPAQPPPQPHPPPPQQQHKEEMAAEAGEA...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
3,P23769,P29590,-,-,Endothelial transcription factor GATA-2,MEVAPEQPRWMAHPAVLNAQHPDSHHPGLAHNYMEPAQLLPPDEVD...,Protein PML,MEPAPARSPRPQQDPARPQEPTMPPPETPSEGRQPSPSPSPTERAP...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
4,P15927,P40763,-,-,Replication protein A 32 kDa subunit,MWNSGFESYGSSSYGGAGGYTQSPGGFGSPAPSQAEKKSRARAQHI...,Signal transducer and activator of transcripti...,MAQWNQLQQLDTRYLEQLHQLYSDSFPMELRQFLAPWIESQDWAYA...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
...,...,...,...,...,...,...,...,...,...,...
2095385,Q13085,Q8NHY2,-,-,Acetyl-CoA carboxylase 1,MDEPSPLAQPLELNQHSRFIIGSVSEDNSEDEISNLVKLDLLEEKE...,E3 ubiquitin-protein ligase COP1,MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSVAVSAAA...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2095386,P53396,Q86T96,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase RNF180,MKRSKELITKNHSQEETSILRCWKCRKCIASSGCFMEYLENQVIKD...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2095387,P53396,Q86TM6,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase synoviolin,MFRTAVMMAASLALTGAVVAHAYYLKHQFYPTVVYLTKSSPSMAVL...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."
2095388,P53396,P46934,-,-,ATP-citrate synthase,MSAKAISEQTGKELLYKFICTTSAIQNRFKYARVTPDTDWARLLQD...,E3 ubiquitin-protein ligase NEDD4,MAQSLRLHFAARRSNTYPLSETSGDDLDSHVHMCFKRPTRISTSNV...,"{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ...","{F, I, G, P, L, C, T, W, M, S, V, D, H, A, N, ..."


In [88]:
ALLOWED = chars

mask_antibodies = biogrid_ppi_unique['check_prot_seq_a'].apply(lambda s: s.issubset(ALLOWED))
mask_target = biogrid_ppi_unique['check_prot_seq_b'].apply(lambda s: s.issubset(ALLOWED))

AA20X = set("ACDEFGHIKLMNPQRSTVWYX")

mask_protein_antibodies = biogrid_ppi_unique['check_prot_seq_a'].apply(
    lambda s: s.issubset(AA20X)
)
mask_protein_target = biogrid_ppi_unique['check_prot_seq_b'].apply(
    lambda s: s.issubset(AA20X)
)

invalid_antibodies = biogrid_ppi_unique[~mask_antibodies]
invalid_target = biogrid_ppi_unique[~mask_target]

antibodies_interactions_1 = biogrid_ppi_unique[mask_antibodies]
antibodies_interactions_2 = biogrid_ppi_unique[mask_target]

In [91]:
antibodies_interactions_1.drop(columns=['check_prot_seq_a', 'check_prot_seq_b'], inplace=True)

/var/folders/_3/p73rghg16cg5d9z62wt7_w1w0000gn/T/ipykernel_7520/2615448566.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  antibodies_interactions_1.drop(columns=['check_prot_seq_a', 'check_prot_seq_b'], inplace=True)


In [92]:
antibodies_interactions_2.drop(columns=['check_prot_seq_a', 'check_prot_seq_b'], inplace=True) 

/var/folders/_3/p73rghg16cg5d9z62wt7_w1w0000gn/T/ipykernel_7520/40455540.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  antibodies_interactions_2.drop(columns=['check_prot_seq_a', 'check_prot_seq_b'], inplace=True)


In [94]:
final_biogrid_ppi_unique = antibodies_interactions_1.merge(antibodies_interactions_2, how="inner")
final_biogrid_ppi_unique.to_csv("final_biogrid_ppi_unique.csv", index=False)